# Lesson 2.4 — Context Isolation with Subagents

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain how subagents isolate context to keep the main loop focused
- Route a subtask to a subagent with its own fresh context
- Return only distilled results to the parent to prevent bloat
- Recognize when isolation beats a single growing context

**You will produce:** A subagent dispatcher that runs an isolated task and returns a compact result.

## Concepts

**Context isolation** gives a subtask its own clean context window so noisy intermediate work never pollutes the main conversation. A **subagent** receives a focused instruction and only the data it needs, does its work in isolation, and returns a distilled result to the parent.

Benefits:
- The parent's context stays small and on-goal (less drift/bloat).
- Each subagent can use a different model tier (e.g., Haiku for extraction, Sonnet for synthesis).
- Failures are contained and easier to debug.

This is the conceptual basis for multi-agent patterns in Phase 4; here we implement the core isolation primitive.

> **Reference:** [Anthropic - Subagents / multi-agent patterns](https://docs.anthropic.com/en/docs/agents-and-tools/agent-sdk/subagents)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Define an isolated subagent

A subagent is a function with its own fresh messages list and focused instruction.

In [ ]:
def subagent(instruction, data, model=None):
    model = model or MODEL_HAIKU
    if not HAVE_API:
        return f"[MOCK subagent result for: {instruction[:40]}]"
    msg = client.messages.create(model=model, max_tokens=200,
        system=instruction,
        messages=[{"role": "user", "content": data}])  # fresh, isolated context
    return msg.content[0].text

result = subagent("Extract all dates. Return a comma-separated list only.",
                  "We met on 2026-01-02 and again on 2026-03-15.")
print(result)

## Code-Along: Step 2 — Parent uses only the distilled result

The parent context receives the compact output, not the subagent's raw work.

In [ ]:
def parent_task(doc):
    dates = subagent("Extract dates as a list only.", doc)
    names = subagent("Extract person names as a list only.", doc)
    # Parent context now holds only two short strings, not the full doc twice
    return {"dates": dates, "names": names}

print(parent_task("Dana met Lee on 2026-05-01 to plan the launch."))

## Exercise

Write `map_reduce(chunks, extract_instruction, reduce_instruction)` that runs a subagent on each chunk (map) then a final subagent to combine the results (reduce). Keep it offline-safe.

In [ ]:
def map_reduce(chunks, extract_instruction, reduce_instruction):
    # TODO: run subagent(extract_instruction, chunk) for each chunk
    # TODO: join the partial results
    # TODO: run subagent(reduce_instruction, joined) and return it
    pass

## Key Takeaways

- Subagents isolate noisy subtasks in their own context window.
- Return only distilled results to the parent to prevent bloat and drift.
- Different subagents can use different model tiers for cost/quality balance.
- Isolation contains failures and simplifies debugging.

---

**Next:** [Lesson 2.5 — Output Handling: Structured Output and Defensive Parsing](./lesson-2.5-structured-output-validation.ipynb)